# Importing Libraries 

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import poisson

In [2]:
PROCESSED_DIR = '../artifacts/processed_data'
MODEL_DIR     = '../artifacts/models'
os.makedirs(MODEL_DIR, exist_ok=True)
 

In [3]:
df = pd.read_csv(f'{PROCESSED_DIR}/poisson_features.csv', parse_dates=['date'])
print(df.shape)
print(df.dtypes)

(49233, 28)
date                         datetime64[us]
home_team                               str
away_team                               str
home_goals                          float64
away_goals                          float64
home_win                              int64
away_win                              int64
draw                                  int64
goal_diff                           float64
result                                int64
is_competitive                         bool
elo_diff                            float64
rank_diff                             int64
neutral                               int64
home_last5_goals_scored             float64
home_last5_goals_conceded           float64
away_last5_goals_scored             float64
away_last5_goals_conceded           float64
diff_last5_goals_scored             float64
diff_last5_goals_conceded           float64
home_last5_win_rate                 float64
away_last5_win_rate                 float64
home_h2h_win_rate   

In [4]:
df.head()

,date,home_team,away_team,home_goals,away_goals,home_win,away_win,draw,goal_diff,result,...,diff_last5_goals_scored,diff_last5_goals_conceded,home_last5_win_rate,away_last5_win_rate,home_h2h_win_rate,away_h2h_win_rate,home_h2h_goal_diff,home_is_host,away_is_host,home_tournament_stage
0,1872-11-30,Scotland,England,0.0,0.0,0,0,1,0.0,1,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,0,0
1,1873-03-08,England,Scotland,4.0,2.0,1,0,0,2.0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,0,0
2,1874-03-07,Scotland,England,2.0,1.0,1,0,0,1.0,0,...,-1.000000,1.000000,0.000000,0.500000,0.000000,0.500000,-2.0,0,0,0
3,1875-03-06,England,Scotland,2.0,2.0,0,0,1,0.0,1,...,0.333333,-0.333333,0.333333,0.333333,0.333333,0.333333,1.0,0,0,0
4,1876-03-04,Scotland,England,3.0,0.0,1,0,0,3.0,0,...,-0.250000,0.250000,0.250000,0.250000,0.250000,0.250000,-1.0,0,0,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49233 entries, 0 to 49232
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       49233 non-null  datetime64[us]
 1   home_team                  49233 non-null  str           
 2   away_team                  49233 non-null  str           
 3   home_goals                 49233 non-null  float64       
 4   away_goals                 49233 non-null  float64       
 5   home_win                   49233 non-null  int64         
 6   away_win                   49233 non-null  int64         
 7   draw                       49233 non-null  int64         
 8   goal_diff                  49233 non-null  float64       
 9   result                     49233 non-null  int64         
 10  is_competitive             49233 non-null  bool          
 11  elo_diff                   49233 non-null  float64       
 12  rank_diff      

In [6]:
META_COLS   = ['date', 'home_team', 'away_team']
TARGET_COLS = ['home_goals', 'away_goals', 'home_win', 'draw', 'away_win', 'result']
 
FEATURE_COLS = [c for c in df.columns if c not in META_COLS + TARGET_COLS]
print(f"\nFeatures used: {len(FEATURE_COLS)}")
print(FEATURE_COLS)


Features used: 19
['goal_diff', 'is_competitive', 'elo_diff', 'rank_diff', 'neutral', 'home_last5_goals_scored', 'home_last5_goals_conceded', 'away_last5_goals_scored', 'away_last5_goals_conceded', 'diff_last5_goals_scored', 'diff_last5_goals_conceded', 'home_last5_win_rate', 'away_last5_win_rate', 'home_h2h_win_rate', 'away_h2h_win_rate', 'home_h2h_goal_diff', 'home_is_host', 'away_is_host', 'home_tournament_stage']


In [7]:
if 'is_competitive' not in df.columns:
    df['is_competitive'] = True
 
train_df = df[df['is_competitive'] & (df['date'] >= '2000-01-01')].copy()
 
train = train_df[train_df['date'] <  '2024-01-01'].copy()
valid = train_df[train_df['date'] >= '2024-01-01'].copy()
 
print(f"\nTrain : {len(train):,} matches  ({train['date'].dt.year.min()}–{train['date'].dt.year.max()})")
print(f"Valid : {len(valid):,} matches  ({valid['date'].dt.year.min()}–{valid['date'].dt.year.max()})")


Train : 14,855 matches  (2000–2023)
Valid : 1,776 matches  (2024–2026)


In [8]:
X_train = train[FEATURE_COLS].fillna(0)
X_valid = valid[FEATURE_COLS].fillna(0)
 
y_train_home = train['home_goals']
y_train_away = train['away_goals']
y_valid_home = valid['home_goals']
y_valid_away = valid['away_goals']

In [9]:
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  
X_valid_scaled = scaler.transform(X_valid) 